In [1]:
# pip install sentence-transformers

In [2]:
from tqdm import tqdm


In [3]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
import json

In [4]:
df = pd.read_csv("/home/ubuntu/datasets/Combined_Dataset_Annotations - Combined_Dataset.csv")
df.head()

,id,soure,subreddit,title,body,created_utc,url,Tags
0,1ljxynj,abortion,abortion,Complications after abortion?,"Hi everyone, Ive read that abortions don’t cau...",2025-06-25 5:59:34,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Sexual, Pregnancy"
1,1ljxtt8,NaN,abortion,Second MA abortion today and I'm absolutely te...,I'm having my second MA abortion today and I'm...,2025-06-25 5:51:08,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Pregnancy, Mental Health"
2,1ljwhkb,abortion,abortion,Help needed/ live in Texas where abortion in b...,Anyone know of a legit site to support women i...,2025-06-25 4:31:43,https://www.reddit.com/r/abortion/comments/1lj...,"Discrimination, Pregnancy"
3,1ljvy6u,NaN,abortion,medical abortion at 6 weeks,I’ll be doing my procedure on Friday and I got...,2025-06-25 4:02:28,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Pregnancy"
4,1ljv5k4,abortion,abortion,Idk what to feel about my decision after doing...,I just had medical abortion yesterday. I was a...,2025-06-25 3:19:54,https://www.reddit.com/r/abortion/comments/1lj...,"Pregnancy, Mental Health"


In [5]:
df['full_text'] = df['title'].fillna('') + '. ' + df['body'].fillna('')

In [7]:
model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(df['full_text'].tolist(), convert_to_tensor=True, show_progress_bar=True)
torch.save(embeddings, "post_embeddings_75.pt")

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
torch.save(embeddings, "post_embeddings.pt")

In [8]:
embeddings = torch.load("post_embeddings_300.pt")

In [9]:
k_max = 3
similar_posts = {}

for i in tqdm(range(len(embeddings)), desc="Computing top-k similar posts"):
    cosine_scores = util.pytorch_cos_sim(embeddings[i], embeddings)[0]
    top_results = torch.topk(cosine_scores, k=k_max + 1)  # +1 to skip self
    indices = top_results.indices.tolist()
    scores = top_results.values.tolist()

    filtered = [(idx, score) for idx, score in zip(indices, scores) if idx != i][:k_max]
    similar_posts[i] = filtered


Computing top-k similar posts: 100%|██████████| 300/300 [00:00<00:00, 3677.28it/s]


In [10]:
with open("/home/ubuntu/similarity_scores/similar_posts_k3_75.json", "w") as f:
    json.dump(similar_posts, f)